# SECTION A

# Install and import dependencies

In [1]:
# Install necessary libraries if not already present
# !pip install langchain langchain-community docx2txt

from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import pandas as pd

# Load and split the document

In [2]:
# 1. Get the directory where this current Python script is located
#current_dir = Path(__file__).parent

# 2. Geat the current working directory (where this Jupyter Notebook is running)
current_dir = Path.cwd()

# 3. Go up to parent directory, and get the file
file_path = current_dir.parent / 'Standard_Operating_Procedure_v1_1.docx'
loader = Docx2txtLoader(file_path)
data = loader.load()

# 4. Initialize the Recursive Character Text Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=80,
    length_function=len,
    add_start_index=True, 
)

# 5. Create the segments (chunks)
# 'data' is the variable from your loader.load()
chunks = text_splitter.split_documents(data)

print(f"Successfully split SOP into {len(chunks)} chunks.")

Successfully split SOP into 13 chunks.


# Preview and Verify

### Symbolic Anchoring: Aligning Chunks with SOP Domains

To ensure the **Evaluator Agent** can perform deterministic audits, we anchor our vector embeddings to the five operational domains defined in **SOP v1.1, Section 3**:

1. **Execution**: Shipping performance and transit metrics.
2. **Planning**: Scheduled vs. Real duration alignment.
3. **Administrative**: Payment verification and transaction types.
4. **Product/Financial**: Profitability and handling constraints.
5. **Temporal/Operational**: Cutoff times and dispatch windows.
6. **Geographical**: Regional delivery limitations and market-specific customs constraints.

By tagging each chunk with a `domain` metadata field, the **n8n RAG Subflow** can use a "Guided Dynamic" filter to retrieve only the rules relevant to the identified SHAP driver.

# SECTION B

In [3]:
# Convert to a DataFrame for easy debugging and inspection
chunk_data = []
for i, chunk in enumerate(chunks):
    chunk_data.append({
        "Chunk_ID": i + 1,
        "Content": chunk.page_content,
        "Metadata": chunk.metadata
    })

df = pd.DataFrame(chunk_data)

# Display the first few chunks to verify the 'semantic bridge' overlap
df.head()

,Chunk_ID,Content,Metadata
0,1,DataCo Standard Operating Procedure (SOP) v1.1...,{'source': 'C:\Users\PC\Documents\Masters Proj...
1,2,3. Inputs\n\nLate delivery assessment is perfo...,{'source': 'C:\Users\PC\Documents\Masters Proj...
2,3,4. Definitions\n\nShipping Delay\nThe differen...,{'source': 'C:\Users\PC\Documents\Masters Proj...
3,4,Order status\n\nDelivery status\n\nLate delive...,{'source': 'C:\Users\PC\Documents\Masters Proj...
4,5,Late deliveries are flagged for further evalua...,{'source': 'C:\Users\PC\Documents\Masters Proj...


# SECTION C

# QDRANT

### Install and Initialize Qdrant

In [4]:
# !pip install fastembed qdrant-client python-docx

import os
# Suppress the Hugging Face symlink warning layout on Windows
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from fastembed import TextEmbedding
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance
import docx

# 1. Initialize Nomic v1.5 (Uses ~300MB RAM)
# FastEmbed handles the download and ONNX optimization automatically
model = TextEmbedding(model_name="nomic-ai/nomic-embed-text-v1.5")

# 2. Setup local Qdrant Connection (Docker)
client = QdrantClient("http://localhost:6333")

collection_name = "dataco_supply_chain_sops"

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

C:\Users\PC\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PC\AppData\Local\Temp\fastembed_cache\models--nomic-ai--nomic-embed-text-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.onnx:   0%|          | 0.00/547M [00:00<?, ?B/s]

2026-06-11 08:21:39.673 | ERROR    | fastembed.common.model_management:download_model:446 - Could not download model from HuggingFace: [WinError 1314] A required privilege is not held by the client: '..\\..\\blobs\\785a8d58b3e2c47f9709d7ac87759a6ac96a106f' -> 'C:\\Users\\PC\\AppData\\Local\\Temp\\fastembed_cache\\models--nomic-ai--nomic-embed-text-v1.5\\snapshots\\e9b6763023c676ca8431644204f50c2b100d9aab\\config.json' Falling back to other sources.
2026-06-11 08:21:39.675 | ERROR    | fastembed.common.model_management:download_model:469 - Could not download model from either source, sleeping for 3.0 seconds, 2 retries left.


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

### Create Collection and Upload

In [5]:
# Recommended practice to avoid deprecation issues
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=768, distance=Distance.COSINE),
    )
    print(f"Collection '{collection_name}' created.")
else:
    print(f"Collection '{collection_name}' already exists.")


Collection 'dataco_supply_chain_sops' created.


In [6]:
# 3. Create Collection with 768 Dimensions for Nomic Matryoshka
from qdrant_client.models import VectorParams, Distance

if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=768, distance=Distance.COSINE),
    )
    print(f"Collection '{collection_name}' created.")
else:
    print(f"Collection '{collection_name}' already exists.")

def ingest_sop_chunks(chunks):
    """
    Processes LangChain chunks, uses a relative vocabulary scoring matrix
    to prevent short or lower-priority domains from being bypassed by early loop breaks.
    """
    documents = [f"search_document: {chunk.page_content}" for chunk in chunks]
    
    # --- OPERATIONAL LEXICON DICTIONARY ---
    sop_lexicon = {
        "Execution": ["execution", "shipping service", "carrier", "transit performance", "delivery performance", "actual performance"],
        "Planning": ["planning", "scheduled", "delivery durations", "fulfillment time", "baseline assessment", "lead times"],
        "Administrative": ["administrative", "type_transfer", "bank transfer", "payment verification", "transaction types", "cash", "debit"],
        "Product/Financial": ["financial", "product", "profit-margin", "profit ratio", "sku", "inventory handling", "pricing"],
        "Temporal/Operational": ["temporal", "ship_hour", "cutoff times", "time-of-day", "dispatch window", "warehouse processing"]
    }
    
    metadata = []
    for chunk in chunks:
        chunk_text_lower = chunk.page_content.lower()
        
        # Calculate matching keyword frequency presence for each domain
        domain_scores = {}
        for domain, keywords in sop_lexicon.items():
            score = sum(1 for kw in keywords if kw in chunk_text_lower)
            if score > 0:
                domain_scores[domain] = score
                
        # Determine the most accurate symbolic anchor based on the highest score
        if domain_scores:
            # Pick the domain with the highest number of unique matching vocabulary words
            assigned_domain = max(domain_scores, key=domain_scores.get)
            
            # GUARDRAIL: If a chunk mentions 4 or more domains simultaneously, 
            # it is an introductory/overview index chunk (like Section 3 Inputs). 
            # We classify it as "General" to prevent false context pooling.
            if len(domain_scores) >= 4:
                assigned_domain = "General"
        else:
            assigned_domain = "General"  # Fallback if no lexicon keys match
                
        metadata.append({
            "text": chunk.page_content, 
            "start_index": chunk.metadata.get("start_index", 0),
            "source": str(file_path.name),
            "domain": assigned_domain  # Stored safely in Qdrant payload
        })
    
    # 3. Generate High-Fidelity Nomic Embeddings
    print(f"Generating Nomic-v1.5 embeddings for {len(documents)} chunks...")
    embeddings = list(model.embed(documents))
    
    # 4. Upload directly to Docker Qdrant Engine
    client.upload_collection(
        collection_name=collection_name,
        vectors=embeddings,
        payload=metadata,
    )
    print(f"✅ SUCCESS: Ingested {len(chunks)} chunks with score-optimized domain payload mapping.")

# Run the ingestion function
ingest_sop_chunks(chunks)

Collection 'dataco_supply_chain_sops' already exists.
Generating Nomic-v1.5 embeddings for 13 chunks...
✅ SUCCESS: Ingested 13 chunks with score-optimized domain payload mapping.
